In [1]:
import torch
import torch.nn as nn
from torch.func import jacrev, jacfwd, vmap, hessian

import sys
sys.path.append("..")

from src.model import *

In [2]:

# Load the trained real-valued model
model = PolyNetwork(
    input_dim=3,
    output_dim=1,
    hidden_dims=[8, 6],
    polynomial_degree=3,
)

# model.load_state_dict(torch.load("./saved_models/polynomial_nn_model.pth"))

# convert to a model compatible with computation over complex space
# model: R^n -> R^m  ==>  cr_model: R^{2n} -> R^{2m} via realification
model_c = CRPolyNetwork.from_polynetwork(model)

# make sure both models use double precision
model = model.double()
model_c = model_c.double()

In [3]:
# sanity check for input
batch_size = 5
input_dim = 3
output_dim = 1
num_hidden_layers = 2
act_degree = 3

x_real = torch.randn(batch_size, input_dim, dtype=torch.double)
y = model(x_real)
print(f"Input of the model has shape: {x_real.shape}.")
print(f"Output of the model has shape: {y.shape}.")

x_imag = torch.zeros_like(x_real)
x_c = c_join(x_real, x_imag)
y_c = model_c(x_c)
print(f"Input of the converted model has shape: {x_c.shape}.")
print(f"Output of the converted model has shape: {y_c.shape}.")

y_real, y_imag = c_split(y_c)
print(f"Real part of the output has shape: {y_real.shape}.")
print(f"Imaginary part of the output has shape: {y_imag.shape}.")

# verify the correctness of the conversion on real inputs
assert torch.allclose(y, y_real) and torch.allclose(y_imag, torch.zeros_like(y_real)), "The converted model does not match the original model on real inputs!"

Input of the model has shape: torch.Size([5, 3]).
Output of the model has shape: torch.Size([5, 1]).
Input of the converted model has shape: torch.Size([5, 6]).
Output of the converted model has shape: torch.Size([5, 2]).
Real part of the output has shape: torch.Size([5, 1]).
Imaginary part of the output has shape: torch.Size([5, 1]).


In [4]:
# sanity check for Jacobian
jac_func = vmap(jacfwd(model))
jac_func_c = vmap(jacfwd(model_c))

J = jac_func(x_real)
J_c = jac_func_c(x_c)

J_real = J_c[:,:output_dim,:input_dim]
J_imag = J_c[:, output_dim:, :input_dim]

print(f"Jacobian of the original model has shape: {J.shape}.")
print(f"Jacobian of the converted model has shape: {J_c.shape}.")

# verify the correctness of the conversion on Jacobians
assert torch.allclose(J_real, J) and torch.allclose(J_imag, torch.zeros_like(J)), "The Jacobian of the converted model does not match the original model on real inputs!"

Jacobian of the original model has shape: torch.Size([5, 1, 3]).
Jacobian of the converted model has shape: torch.Size([5, 2, 6]).


In [5]:
# sanity check for Hessian
hess_func = vmap(hessian(model))
hess_func_c = vmap(hessian(model_c))

H = hess_func(x_real)
H_c = hess_func_c(x_c)

print(f"Hessian of the original model has shape: {H.shape}.")
print(f"Hessian of the converted model has shape: {H_c.shape}.")

H_real = 1/4 * (H_c[:, 0, :input_dim, :input_dim] - H_c[:, 0, input_dim:, input_dim:] + 2 * H_c[:, 1, :input_dim, input_dim:]).unsqueeze(1)
H_imag = 1/4 * (H_c[:, 1, :input_dim, :input_dim] - H_c[:, 1, input_dim:, input_dim:] - 2 * H_c[:, 0, :input_dim, input_dim:]).unsqueeze(1)

# verify the correctness of the conversion on Hessians
assert torch.allclose(H, H_real) and torch.allclose(H_imag, torch.zeros_like(H_real)), "The converted model's Hessian does not match the original model's Hessian on real inputs!"

Hessian of the original model has shape: torch.Size([5, 1, 3, 3]).
Hessian of the converted model has shape: torch.Size([5, 2, 6, 6]).


In [6]:
assert output_dim == 1, "Output dimension must be 1."
assert J_real.shape == (batch_size, output_dim, input_dim)
assert J_imag.shape == (batch_size, output_dim, input_dim)
assert H_real.shape == (batch_size, output_dim, input_dim, input_dim)
assert H_imag.shape == (batch_size, output_dim, input_dim, input_dim)

lmbda_real = torch.rand(batch_size, 1, 1)
lmbda_imag = torch.rand(batch_size, 1, 1)
lmbda_c = c_join(lmbda_real, lmbda_imag)
lmbda_c

H_real = H_real.squeeze(1)
H_imag = H_imag.squeeze(1)

print(J_real.shape)
print(H_imag.shape)

I = torch.eye(input_dim, dtype=torch.double).expand(batch_size, -1, -1)

ll_real = I + lmbda_real * H_real - lmbda_imag * H_imag
lr_real = J_real.transpose(1,2)
ul_real = J_real
ur_real = torch.zeros(batch_size, 1, 1, dtype=torch.double)

J_homo_real = torch.concat(
    [
        torch.concat([ul_real, ur_real], dim=2),
        torch.concat([ll_real, lr_real], dim=2)
    ],
    dim=1
)

ll_imag = lmbda_real * H_imag + lmbda_imag * H_real
lr_imag = J_imag.transpose(1,2)
ul_imag = J_imag
ur_imag = torch.zeros(batch_size, 1, 1, dtype=torch.double)

J_homo_imag = torch.concat(
    [
        torch.concat([ul_imag, ur_imag], dim=2),
        torch.concat([ll_imag, lr_imag], dim=2)
    ],
    dim=1
)

J_homo_c = torch.concat(
    [
        torch.concat([J_homo_real, -J_homo_imag], dim=2),
        torch.concat([J_homo_imag, J_homo_real], dim=2)
    ],
    dim=1
)

print(J_homo_real.shape)
print(J_homo_imag.shape)
print(J_homo_c.shape)

torch.Size([5, 1, 3])
torch.Size([5, 3, 3])
torch.Size([5, 4, 4])
torch.Size([5, 4, 4])
torch.Size([5, 8, 8])


In [7]:
gamma_angle = torch.rand(1) * 2 * torch.pi
gamma_real, gamma_imag = torch.cos(gamma_angle), torch.sin(gamma_angle)
gamma_c = c_join(gamma_real, gamma_imag)
gamma_c

tensor([-0.9761, -0.2172])

In [8]:
xi_real = torch.randn(input_dim, dtype=torch.double).expand(batch_size, -1)
xi_imag = torch.zeros_like(xi_real)
# x_real - xi_real

y_real, y_imag = c_split(model_c(x_c))

F_real = torch.concat([y_real, x_real - xi_real + (lmbda_real * J_real - lmbda_imag * J_imag).squeeze(1)], dim=1)
F_imag = torch.concat([y_imag, x_imag + (lmbda_real * J_imag + lmbda_imag * J_real).squeeze(1)], dim=1)

F = c_join(F_real, F_imag)
print(F.shape)

torch.Size([5, 8])


In [9]:
dims = torch.ones(input_dim + 1) * act_degree ** num_hidden_layers
dims = dims.expand(batch_size, -1)

x_lmbda_real = torch.concat([x_real, lmbda_real.squeeze(2)], dim = 1)
x_lmbda_imag = torch.concat([x_imag, lmbda_imag.squeeze(2)], dim = 1)
rho = torch.hypot(x_lmbda_real, x_lmbda_imag)
theta = torch.atan2(x_lmbda_real, x_lmbda_imag)

rho_pow = torch.pow(rho, dims)
real_part = torch.cos(dims * theta) * rho_pow - 1
imag_part = torch.sin(dims * theta) * rho_pow

G_real = gamma_real * real_part - gamma_imag * imag_part
G_imag = gamma_real * imag_part + gamma_imag * real_part

G = c_join(G_real, G_imag)

torch.linalg.solve(J_homo_c, F - G)

tensor([[-3.2274e+01, -2.5728e+02,  2.9811e+02,  1.3087e+05, -6.2558e+00,
         -4.7167e+01,  5.4726e+01,  2.4214e+04],
        [-1.0064e+02, -2.5617e+02,  9.5489e+01,  6.1780e+04,  3.5407e+01,
          9.9890e+01, -3.9293e+01, -2.4668e+04],
        [ 4.0654e+01, -6.0322e+01,  5.5352e+01,  7.5641e+03,  1.1636e+01,
         -2.7626e+01,  1.5550e+01,  2.0911e+03],
        [-1.2602e+03,  3.1935e+03, -1.9284e+03, -6.8916e+05,  5.9868e+03,
         -1.5113e+04,  9.1562e+03,  3.2505e+06],
        [ 1.2110e+02, -1.1792e+02,  5.5347e+01,  3.0507e+04, -5.9364e+01,
         -9.9295e+01, -2.8903e+01, -1.4525e+04]], dtype=torch.float64,
       grad_fn=<LinalgSolveExBackward0>)